In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.ticker import ScalarFormatter
import matplotlib.cm as cm
import numpy as np

n_panels = 5

cmap = cm.BrBG

####
# Get the original BrBG colormap
original_cmap = plt.get_cmap('BrBG')

# Sample 256 values from the original colormap
n_colors = 256
colors = original_cmap(np.linspace(0, 1, n_colors))

# Identify midpoint (assumes symmetric colormap)
midpoint = n_colors // 2

# Replace the midpoint with white
colors[midpoint] = [1.0, 1.0, 1.0, 1.0]  # RGBA white

# Create new colormap
new_cmap = LinearSegmentedColormap.from_list("BrBG_white0", colors)
####

alpha = 0.2

# Figure with outer grid for each panel
fig = plt.figure(figsize=(7 * n_panels, 7), constrained_layout=False)
#plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # reserve room for colorbars

outer = gridspec.GridSpec(1, n_panels, wspace=0.2)

for i in range(n_panels):
    # Create inner grid for each panel
    inner_gs = gridspec.GridSpecFromSubplotSpec(
        2, 2, subplot_spec=outer[i],
        #width_ratios=[5, 2], height_ratios=[2, 5],
        width_ratios=[4, 1], height_ratios=[1, 4],
        wspace=0.05, hspace=0.05
    )

    # compute spectrum min and max for y-axis bounds
    spectrum_lat_min = min(np.min(wavesimsv.m1_marginal_lat[0,0,:,:].numpy()), np.min(wavesimsv.m2_marginal_lat[0,0,:,:].numpy()))
    spectrum_lat_max = max(np.max(wavesimsv.m1_marginal_lat[0,0,:,:].numpy()), np.max(wavesimsv.m2_marginal_lat[0,0,:,:].numpy()))
    spectrum_lon_min = min(np.min(wavesimsv.m1_marginal_lon[0,0,:,:].numpy()), np.min(wavesimsv.m2_marginal_lon[0,0,:,:].numpy()))
    spectrum_lon_max = max(np.max(wavesimsv.m1_marginal_lon[0,0,:,:].numpy()), np.max(wavesimsv.m2_marginal_lon[0,0,:,:].numpy()))

    # Subplot axes
    ax_main = fig.add_subplot(inner_gs[1, 0], projection=ccrs.PlateCarree())
    #ax_top = fig.add_subplot(inner_gs[0, 0]) #, sharex=ax_main)
    #ax_right = fig.add_subplot(inner_gs[1, 1]) #, sharey=ax_main)

    # Top marginal (longitude)
    # --- Add top marginal inset ---
    ax_top = inset_axes(ax_main, width="100%", height="100%", loc='upper center',
                        bbox_to_anchor=(0, 1.05, 1, 0.3), bbox_transform=ax_main.transAxes, borderpad=0)
    ax_top.set_xlim(ax_main.get_xlim())
    ax_top.plot(wavesimsv.m1_marginal_lon[0,0,:,i], color=cmap(0.75), label = 'MSD (Reference)')
    ax_top.fill_between(np.arange(len(lon)), wavesimsv.m1_marginal_lon[0,0,:,i], color=cmap(0.75), alpha=alpha)
    ax_top.plot(wavesimsv.m2_marginal_lon[0,0,:,i], color=cmap(0.25), label = 'MSD (Perturbed)')
    ax_top.fill_between(np.arange(len(lon)), wavesimsv.m2_marginal_lon[0,0,:,i], color=cmap(0.25), alpha=alpha)
    ax_top.plot(0.5 * (wavesimsv.m1_marginal_lon[0,0,:,i] + wavesimsv.m2_marginal_lon[0,0,:,i]), linestyle='--', color='purple', label = 'Mixture')
    #ax_top.fill_between(np.arange(len(lon)), 0.5 * (wavesimsv.map1_lon_marginal[0,0,:,i] + wavesimsv.map2_lon_marginal[0,0,:,i]), color='purple', alpha=alpha * 0.5)
    #ax_top.axvline(x=wavesimsv.map1_multi_scale_com_lon[0,0,i], color=cmap(0.75), linestyle='--', label=f'Center of Mass (Reference)')
    #ax_top.axvline(x=wavesimsv.map2_multi_scale_com_lon[0,0,i], color=cmap(0.25), linestyle='--', label=f'Center of Mass (Perturbed)')
    #ax_top.annotate(f'CD={wavesimsv.centroid_dist_lon[0,0,i]:.2f}', xy=(0,1), xycoords='axes fraction',
    #                xytext=(5, -5), textcoords='offset points', ha='left', va='top')
    ax_top.set_xticks([])
    ax_top.set_xlim(0, wavesimsv.W-1)
    ax_top.set_ylim(0, spectrum_lon_max)
    ax_top.set_yticks([np.round(num, 2) for num in np.linspace(0, spectrum_lon_max, 4)])
    #ax_top.yaxis.set_label_position("right")
    ax_top.set_frame_on(True)

    # Right marginal (latitude)
    ax_right = inset_axes(ax_main, width="100%", height="100%", loc='center right',
                        bbox_to_anchor=(1.07, 0, 0.3, 1), bbox_transform=ax_main.transAxes, borderpad=0)
    ax_right.set_ylim(ax_main.get_ylim())
    ax_right.plot(wavesimsv.m1_marginal_lat[0,0,:,i], np.arange(len(lat)), color=cmap(0.75), label = 'MSD (Reference)')
    ax_right.fill_betweenx(np.arange(len(lat)), wavesimsv.m1_marginal_lat[0,0,:,i], color=cmap(0.75), alpha=alpha)
    ax_right.plot(wavesimsv.m2_marginal_lat[0,0,:,i], np.arange(len(lat)), color=cmap(0.25), label = 'MSD (Perturbed)')
    ax_right.fill_betweenx(np.arange(len(lat)), wavesimsv.m2_marginal_lat[0,0,:,i], color=cmap(0.25), alpha=alpha)
    ax_right.plot(0.5 * (wavesimsv.m1_marginal_lat[0,0,:,i] + wavesimsv.m2_marginal_lat[0,0,:,i]), np.arange(len(lat)), linestyle='--', color='purple', label = 'Mixture')
    #ax_right.axhline(y=wavesimsv.map1_multi_scale_com_lat[0,0,i], color=cmap(0.75), linestyle='--')
    #ax_right.axhline(y=wavesimsv.map2_multi_scale_com_lat[0,0,i], color=cmap(0.25), linestyle='--')
    #ax_right.annotate(f'CD={wavesimsv.centroid_dist_lat[0,0,i]:.2f}', xy=(0,1), xycoords='axes fraction',
    #                  xytext=(20, -5), textcoords='offset points', ha='left', va='top')
    ax_right.set_yticks([])
    ax_right.set_ylim(0, wavesimsv.H-1)
    ax_right.set_xlim(0, spectrum_lat_max)
    ax_right.set_xticks([np.round(num, 2) for num in np.linspace(0, spectrum_lat_max, 4)])
    #ax_right.yaxis.set_label_position("left")
    ax_right.set_frame_on(True)

    # Main panel
    gridline_settings = {'draw_labels': True, 'linewidth': 0.0, 
                        'color': 'gray', 'alpha': 0.5, 'linestyle': '--'}
    

    ax_main.set_extent(domain_extent, crs=ccrs.PlateCarree())
    ax_main.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.5, zorder=2)
    coeff_bias = (wavesimsv.m1_coeffs[0,0,:,:,i] - wavesimsv.m2_coeffs[0,0,:,:,i])
    norm = mcolors.TwoSlopeNorm(vmin=coeff_bias.min(), vcenter=0.0, vmax=coeff_bias.max())
    coeff_diff_im = ax_main.pcolormesh(lon_grid, lat_grid, coeff_bias, cmap=cmap, norm=norm, shading='auto', transform=ccrs.PlateCarree())

    #ax_main.set_aspect('auto')  # or 'equal'
    #ax_main.set_adjustable('datalim')

    # Manually add a colorbar Axes (absolute positioning)
    #cbar_ax = fig.add_axes([0.125, 0.05, 0.6, 0.02])  # [left, bottom, width, height] in figure fraction
    #cbar = plt.colorbar(coeff_diff_im, cax=cbar_ax, orientation='horizontal', format=ScalarFormatter())
    #cbar.set_label('Precipitation anomaly [mm]', fontsize=10)

    cax = inset_axes(ax_main, width="100%", height="5%", loc='lower center',
                 bbox_to_anchor=(0, -0.15, 1, 1),
                 bbox_transform=ax_main.transAxes, borderpad=0)
    
    cbar = plt.colorbar(coeff_diff_im, cax=cax, orientation='horizontal', format=ScalarFormatter())
    cbar.set_label('Precipitation difference (mm)', fontsize=10)
    cbar.ax.tick_params(labelsize=10)

    gl = ax_main.gridlines(**gridline_settings)
    gl.top_labels = False
    gl.right_labels = False

    # legend
    #h1, l1 = ax_top.get_legend_handles_labels()
    #ax_top.legend(h1, l1, loc='center', bbox_to_anchor=(1.2, 0.5), fancybox=False, shadow=False, ncol=1, fontsize=8)
    #ax_right.legend().remove()

# legend
h1, l1 = ax_top.get_legend_handles_labels()
plt.legend(h1, l1, loc='center', bbox_to_anchor=(-2.5, -4), fancybox=False, shadow=False, ncol=5, fontsize=12)
#plt.legend().remove()

plt.show()